In [33]:
import pandas as pd
import numpy as np
from scipy import stats

In [7]:
histDesign=pd.read_csv(r'C:\Users\shapa\OneDrive\Рабочий стол\hw-task-2-dataset\[historical]-A-B-test-new-checkout-design.csv')
expDesign=pd.read_csv(r'C:\Users\shapa\OneDrive\Рабочий стол\hw-task-2-dataset\[experimental]-A-B-test-new-checkout-design.csv')

Approach
I used conversion rate as the metric and a z-test for two proportions, with α = 0.05 and power = 0.8, and calculated the MDE from the historical data to see what size of effect each test could detect. Before trusting any result, I checked validity with an A/A test on the historical data and an SRM test on the group sizes. Then we ran the A/B test on the experimental data, looked at the p-value and the 95% confidence interval of the lift, and made the release decision.

Cleaning Historical Data

In [9]:
histDesign.head()

,Unnamed: 0,user_id,user_type,is_converted,date
0,0,ChFMPfpxFFdTYnCr,A,1,2023-06-03 00:00:00.000000000
1,1,EXAKRlP5fyfDDYIv,A,0,2023-06-03 00:13:08.289552837
2,2,razGY3Lgq3kmFXvp,A,1,2023-06-03 00:26:16.579105674
3,3,bB6CeJy78TxltOc0,B,1,2023-06-03 00:39:24.868658512
4,4,07uCZyMnNgG4iA6M,A,1,2023-06-03 00:52:33.158211349


In [19]:
histDesign=histDesign.drop(columns=['Unnamed: 0'])  #Dropping colomn 'Unnamed'

In [21]:
histDesign.head()

,user_id,user_type,is_converted,date
0,ChFMPfpxFFdTYnCr,A,1,2023-06-03 00:00:00.000000000
1,EXAKRlP5fyfDDYIv,A,0,2023-06-03 00:13:08.289552837
2,razGY3Lgq3kmFXvp,A,1,2023-06-03 00:26:16.579105674
3,bB6CeJy78TxltOc0,B,1,2023-06-03 00:39:24.868658512
4,07uCZyMnNgG4iA6M,A,1,2023-06-03 00:52:33.158211349


In [31]:
histDesign.shape

(9975, 4)

In [23]:
histDesign.isna().sum()

user_id         0
user_type       0
is_converted    0
date            0
dtype: int64

In [27]:
histDesign.describe() 

,is_converted
count,9975.000000
mean,0.256441
std,0.436690
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [49]:
histDesign.dtypes

user_id         object
user_type       object
is_converted     int64
date            object
dtype: object

In [71]:
histDesign['date']=pd.to_datetime(histDesign["date"])

In [53]:
histDesign.dtypes

user_id                 object
user_type               object
is_converted             int64
date            datetime64[ns]
dtype: object

In [57]:
print(histDesign["user_id"].duplicated().sum())

0


In [59]:
histDesign['user_type'].value_counts()

user_type
B    4995
A    4980
Name: count, dtype: int64

In [371]:
print("Date range:")
print(histDesign["date"].min())
print(histDesign["date"].max())
histDesign['day'] = (histDesign['date'] - histDesign['date'].min()).dt.days
total_days = histDesign['day'].max() + 1
print("Historical days:", total_days, "=> roughly", total_days // 7, "weeks")

Date range:
2023-06-03 00:00:00
2023-09-02 00:00:00
Historical days: 92 => roughly 13 weeks


Cleaning Experimental Data

In [69]:
expDesign.head()

,Unnamed: 0,user_id,user_type,is_converted,date
0,0,VpaxSu4yMhWoEPge,A,0,2023-09-03 00:00:00.000000000
1,1,KKiS4VwXJE2f1qTs,A,0,2023-09-03 00:12:46.390486402
2,2,JCvZx8u01thQJ9Qe,B,0,2023-09-03 00:25:32.780972804
3,3,GG62HQ4zJ4EhWxep,B,0,2023-09-03 00:38:19.171459206
4,4,H2OgGAfY9D69dlgl,A,0,2023-09-03 00:51:05.561945608


In [73]:
expDesign=expDesign.drop(columns=['Unnamed: 0'])

In [75]:
expDesign.dtypes

user_id         object
user_type       object
is_converted     int64
date            object
dtype: object

In [77]:
expDesign['date']=pd.to_datetime(expDesign["date"])

In [79]:
expDesign.head()

,user_id,user_type,is_converted,date
0,VpaxSu4yMhWoEPge,A,0,2023-09-03 00:00:00.000000000
1,KKiS4VwXJE2f1qTs,A,0,2023-09-03 00:12:46.390486402
2,JCvZx8u01thQJ9Qe,B,0,2023-09-03 00:25:32.780972804
3,GG62HQ4zJ4EhWxep,B,0,2023-09-03 00:38:19.171459206
4,H2OgGAfY9D69dlgl,A,0,2023-09-03 00:51:05.561945608


In [87]:
expDesign.shape

(3157, 4)

In [93]:
expDesign.isna().sum()

user_id         0
user_type       0
is_converted    0
date            0
dtype: int64

In [95]:
print(expDesign["user_id"].duplicated().sum())

0


In [83]:
expDesign['user_type'].value_counts()

user_type
B    1579
A    1578
Name: count, dtype: int64

In [375]:
print("Date range:")
print(expDesign["date"].min())
print(expDesign["date"].max())
expDesign['day'] = (expDesign['date'] - expDesign['date'].min()).dt.days
total_days = expDesign['day'].max() + 1
print("Experimental days:", total_days, "=> roughly", total_days // 7, "weeks")

Date range:
2023-09-03 00:00:00
2023-09-30 23:52:08.375085291
Experimental days: 28 => roughly 4 weeks


In [97]:
print(histDesign["is_converted"].isin([0, 1]).all())
print(expDesign["is_converted"].isin([0, 1]).all())

True
True


AA and AB testing on Datasets

In [101]:
alpha = 0.05              #threshold for rejection 
power = 0.8               # chance to detect a real effect
alpha_chisquare = 0.001   # for SRM test

In [363]:
# we apply z test because sample sizes are more than 30

In [325]:
def group_stats(df):
    return df.groupby('user_type')['is_converted'].agg(['sum', 'count', 'mean']) #values needed to future test formulas

def z_test(g):
    "Z-test for two proportions. g = output of group_stats."
    n_a, n_b = g.loc['A', 'count'], g.loc['B', 'count']
    p_a, p_b = g.loc['A', 'mean'], g.loc['B', 'mean']
    p_combined = (g.loc['A', 'sum'] + g.loc['B', 'sum']) / (n_a + n_b)
    se = np.sqrt(p_combined * (1 - p_combined) * (1 / n_a + 1 / n_b)) # standard error
    z = (p_b - p_a) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value

def confidence_interval(g):
    "95% CI for the lift (B - A). Returns lift, real lift: low, high."
    n_a, n_b = g.loc['A', 'count'], g.loc['B', 'count']
    p_a, p_b = g.loc['A', 'mean'], g.loc['B', 'mean']
    se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    lift = p_b - p_a
    return lift, lift - z_crit * se, lift + z_crit * se  #gap ± 1.96 × se

def mde_formula(p, n_total):
    z_total = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    return z_total * np.sqrt(4 * p * (1 - p) / n_total)

def mde_by_week(exp, p_base):          # <-- p_base is now an argument
    days = (exp['date'] - exp['date'].min()).dt.days
    exp = exp.assign(week=days // 7 + 1)

    rows = []
    for w in sorted(exp['week'].unique()):
        n_this_week = (exp['week'] == w).sum()          # users in this single week
        n_cumulative = (exp['week'] <= w).sum()         # users from week 1 up to this week
        rows.append({
            'week': w,
            'Users': n_this_week,
            'MDE(pp)': round(mde_formula(p_base, n_this_week) * 100, 2),
            'Users so far': n_cumulative,
            'MDE so far(pp)': round(mde_formula(p_base, n_cumulative) * 100, 2),
        })
    out = pd.DataFrame(rows).set_index('week')
    out['MDE so far(relative, %)'] = (out['MDE so far(pp)'] / (p_base * 100) * 100).round(1)
    return out

# H0: p_A == p_B   (new version does not change conversion)
# H1: p_A != p_B   (new version changes conversion)

Testing on New Checkout Design

In [331]:
p_base = histDesign['is_converted'].mean()
print('CHECKOUT baseline:', round(p_base * 100, 2), '%')
print(mde_by_week(expDesign, p_base))

# --- A/A test (historical data)
g_hist = group_stats(histDesign)
print(g_hist)
z, aa_p = z_test(g_hist)
print("A/A p_value:", round(aa_p, 4), "| significant?", aa_p <= alpha)       

# --- SRM test (experimental data)
sizes = expDesign['user_type'].value_counts().values
srm_p = stats.chisquare(sizes)[1]
print("SRM p_value:", round(srm_p, 4), "| problem?", srm_p <= alpha_chisquare) 

# --- A/B test (experimental data)
print('\n')
g_exp = group_stats(expDesign)
print(g_exp)
z, ab_p = z_test(g_exp)
lift, ci_low, ci_high = confidence_interval(g_exp)
print("A/B p_value:", round(ab_p, 4), "| significant?", ab_p <= alpha)
print("Lift:", round(lift * 100, 2), "pp | 95% CI: [", round(ci_low * 100, 2), ",", round(ci_high * 100, 2), "] pp")


CHECKOUT baseline: 25.64 %
      Users  MDE(pp)  Users so far  MDE so far(pp)  MDE so far(relative, %)
week                                                                       
1       790     8.71           790            8.71                     34.0
2       789     8.71          1579            6.16                     24.0
3       789     8.71          2368            5.03                     19.6
4       789     8.71          3157            4.35                     17.0
            sum  count      mean
user_type                       
A          1273   4980  0.255622
B          1285   4995  0.257257
A/A p_value: 0.8517 | significant? False
SRM p_value: 0.9858 | problem? False


           sum  count      mean
user_type                      
A          386   1578  0.244613
B          448   1579  0.283724
A/B p_value: 0.0127 | significant? True
Lift: 3.91 pp | 95% CI: [ 0.84 , 6.98 ] pp


Decision about release of the checkout design 

In [311]:
#the 4-week experiment could reliably detect a lift of about 4.4 pp or larger
#The p-value is 0.85, which is above 0.05, so we fail to reject H0.
#For SRM test the p-value is 0.99, which is above 0.001. So the split was fair.
#For A/B test The p-value is 0.0127, which is below 0.05, so we reject H0: 
#the new checkout does change conversion, and since B is higher, we should RELEASE a new checkout design.

Color Palette Data Analysis

Dataset Preparation

In [124]:
histColorPal=pd.read_csv(r'C:\Users\shapa\OneDrive\Рабочий стол\hw-task-2-dataset\[historical]-A-B-test-new-color-palette.csv')
expColorPal=pd.read_csv(r'C:\Users\shapa\OneDrive\Рабочий стол\hw-task-2-dataset\[experimental]-A-B-test-new-color-palette.csv')

In [126]:
histColorPal.head()

,Unnamed: 0,user_id,user_type,is_converted,date
0,0,lBB4yig3YoFkauzu,B,1,2023-06-03 00:00:00.000000000
1,1,wm9wCAi3zRYkvUxX,B,0,2023-06-03 00:12:47.962492674
2,2,OV02EQ5EHNchvdr8,B,0,2023-06-03 00:25:35.924985348
3,3,9KEO8nTnFH1ufANr,B,1,2023-06-03 00:38:23.887478023
4,4,tF7OJ4WSV2VKER6C,B,1,2023-06-03 00:51:11.849970697


In [128]:
histColorPal=histColorPal.drop(columns=['Unnamed: 0'])
expColorPal=expColorPal.drop(columns=['Unnamed: 0'])

In [130]:
expColorPal.head()

,user_id,user_type,is_converted,date
0,gR58p90ROF7AoUtB,A,0,2023-09-03 00:00:00.000000000
1,zD5XYTeYDxLuxASS,A,0,2023-09-03 00:12:27.802929427
2,P8s5PhJEwCA7KNcq,A,0,2023-09-03 00:24:55.605858854
3,u6hL7RKpjzXSR3ek,B,0,2023-09-03 00:37:23.408788282
4,9UIZcnWkYjSNGjAr,B,0,2023-09-03 00:49:51.211717709


In [136]:
histColorPal['date']=pd.to_datetime(histColorPal["date"])
expColorPal['date']=pd.to_datetime(expColorPal["date"])

In [176]:
print(f"The shape of Historical Data: {histColorPal.shape}\n")
print(f"The shape of Experimental Data: {expColorPal.shape}\n")
print(histColorPal.dtypes)

The shape of Historical Data: (10239, 4)

The shape of Experimental Data: (3236, 4)

user_id                 object
user_type               object
is_converted             int64
date            datetime64[ns]
dtype: object


In [168]:
print(f"Amount of NULL values in historical data: \n{histColorPal.isna().sum()}\n")
print(f"Amount of NULL values in experimental data: \n{expColorPal.isna().sum()}\n")
print(f"Number of duplicated values in Historical: {histColorPal["user_id"].duplicated().sum()}")
print(f"Number of duplicated values in Experimental: {expColorPal["user_id"].duplicated().sum()}")
print("Date range:")
print(histColorPal["date"].min())
print(expColorPal["date"].max())

Amount of NULL values in historical data: 
user_id         0
user_type       0
is_converted    0
date            0
dtype: int64

Amount of NULL values in experimental data: 
user_id         0
user_type       0
is_converted    0
date            0
dtype: int64

Number of duplicated values in Historical: 0
Number of duplicated values in Experimental: 0
Date range:
2023-06-03 00:00:00
2023-09-30 23:59:02.476697736


Testing on new Color Palette data

In [339]:
p_base = histColorPal['is_converted'].mean()
print('Palette baseline:', round(p_base * 100, 2), '%')
print(mde_by_week(expColorPal, p_base))

# --- A/A test (historical data)
g_hist = group_stats(histColorPal)
print(g_hist)
z, aa_p = z_test(g_hist)
print("A/A p_value:", round(aa_p, 4), "| significant?", aa_p <= alpha)       

# --- SRM test (experimental data)
sizes = expColorPal['user_type'].value_counts().values
srm_p = stats.chisquare(sizes)[1]
print("SRM p_value:", round(srm_p, 4), "| problem?", srm_p <= alpha_chisquare) 

# --- A/B test (experimental data)
print('\n')
g_exp = group_stats(expColorPal)
print(g_exp)
z, ab_p = z_test(g_exp)
lift, ci_low, ci_high = confidence_interval(g_exp)
print("A/B p_value:", round(ab_p, 4), "| significant?", ab_p <= alpha)
print("Lift:", round(lift * 100, 2), "pp | 95% CI: [", round(ci_low * 100, 2), ",", round(ci_high * 100, 2), "] pp")

Palette baseline: 24.48 %
      Users  MDE(pp)  Users so far  MDE so far(pp)  MDE so far(relative, %)
week                                                                       
1       809     8.47           809            8.47                     34.6
2       809     8.47          1618            5.99                     24.5
3       809     8.47          2427            4.89                     20.0
4       809     8.47          3236            4.23                     17.3
            sum  count      mean
user_type                       
A          1294   5123  0.252586
B          1212   5116  0.236904
A/A p_value: 0.065 | significant? False
SRM p_value: 0.7785 | problem? False


           sum  count      mean
user_type                      
A          415   1626  0.255228
B          399   1610  0.247826
A/B p_value: 0.6276 | significant? False
Lift: -0.74 pp | 95% CI: [ -3.73 , 2.25 ] pp


Decision about release of the color palette update

In [344]:
#the 4-week experiment could reliably detect a change of about 4.2 pp or larger.
#A/A test. A converted 25.26% and B 23.69% before the change. The p-value is 0.065, above 0.05, so we fail to reject H0 and the test passes
#SRM test.The p-value is 0.78, above 0.001. There's no sample ratio mismatch.
#A/B test.The p-value is 0.63, far above 0.05, so we FAIL to reject H0: there's no evidence the new palette changes conversion.
#Do not release the new color palette as a conversion improvement.

Ml algorithm recommendation dataset Analysis

Dataset Preparation

In [353]:
def overview(dataset):
    print(f"Shape of dataset: {dataset.shape}")
    print('\n')
    dataset['date']=pd.to_datetime(dataset["date"])
    print('Data Types:')
    print(dataset.dtypes)
    print('NULL VALUES:')
    print(dataset.isna().sum())
    print(f"Number of duplicated values: {dataset["user_id"].duplicated().sum()}\n")
    print("Date range:")
    print(dataset["date"].min())
    print(dataset["date"].max())

In [355]:
histML=pd.read_csv(r'C:\Users\shapa\OneDrive\Рабочий стол\hw-task-2-dataset\[historical]-A-B-test-new-ml-recommendation-system.csv')
expML=pd.read_csv(r'C:\Users\shapa\OneDrive\Рабочий стол\hw-task-2-dataset\[experimental]-A-B-test-new-ml-recommendation-system.csv')

In [357]:
histML.head()

,Unnamed: 0,user_id,user_type,is_converted,date
0,0,9ztOVjUNMcvqqcWC,A,0,2023-06-03 00:00:00.000000000
1,1,3fWUE6vES1JN2f1V,A,0,2023-06-03 00:12:04.311377245
2,2,VLWwJTPipF7UkM48,B,0,2023-06-03 00:24:08.622754491
3,3,yPuOmjnOsntSVwvu,B,1,2023-06-03 00:36:12.934131736
4,4,LAkC0xqFSrI8Jrwi,B,0,2023-06-03 00:48:17.245508982


In [359]:
print(overview(histML))

Shape of dataset: (10856, 5)


Data Types:
Unnamed: 0               int64
user_id                 object
user_type               object
is_converted             int64
date            datetime64[ns]
dtype: object
NULL VALUES:
Unnamed: 0      0
user_id         0
user_type       0
is_converted    0
date            0
dtype: int64
Number of duplicated values: 0

Date range:
2023-06-03 00:00:00
2023-09-02 00:00:00
None


In [225]:
expML.head()

,Unnamed: 0,user_id,user_type,is_converted,date
0,0,qEE4zvFtsRXn7RYH,A,0,2023-09-03 00:00:00.000000000
1,1,tWZIqUuTmLyT1bAf,A,0,2023-09-03 00:12:12.476243711
2,2,oLkmjxUH6woDgcm2,B,0,2023-09-03 00:24:24.952487423
3,3,W3zzcfdjnURwFu0i,B,1,2023-09-03 00:36:37.428731134
4,4,3gZyzq2RtPOOdHvX,A,0,2023-09-03 00:48:49.904974846


In [231]:
expML.drop('Unnamed: 0', axis=1,inplace=True)

In [361]:
print(overview(expML))

Shape of dataset: (3303, 5)


Data Types:
Unnamed: 0               int64
user_id                 object
user_type               object
is_converted             int64
date            datetime64[ns]
dtype: object
NULL VALUES:
Unnamed: 0      0
user_id         0
user_type       0
is_converted    0
date            0
dtype: int64
Number of duplicated values: 0

Date range:
2023-09-03 00:00:00
2023-09-30 23:50:36.556735606
None


Testing on ML dataset

In [337]:
p_base = histML['is_converted'].mean()
print('ML baseline:', round(p_base * 100, 2), '%')
print(mde_by_week(expML, p_base))

# --- A/A test (historical data)
g_hist = group_stats(histML)
print(g_hist)
z, aa_p = z_test(g_hist)
print("A/A p_value:", round(aa_p, 4), "| significant?", aa_p <= alpha)       

# --- SRM test (experimental data)
sizes = expML['user_type'].value_counts().values
srm_p = stats.chisquare(sizes)[1]
print("SRM p_value:", round(srm_p, 4), "| problem?", srm_p <= alpha_chisquare) 

# --- A/B test (experimental data)
print('\n')
g_exp = group_stats(expML)
print(g_exp)
z, ab_p = z_test(g_exp)
lift, ci_low, ci_high = confidence_interval(g_exp)
print("A/B p_value:", round(ab_p, 4), "| significant?", ab_p <= alpha)
print("Lift:", round(lift * 100, 2), "pp | 95% CI: [", round(ci_low * 100, 2), ",", round(ci_high * 100, 2), "] pp")

ML baseline: 28.28 %
      Users  MDE(pp)  Users so far  MDE so far(pp)  MDE so far(relative, %)
week                                                                       
1       826     8.78           826            8.78                     31.0
2       826     8.78          1652            6.21                     22.0
3       826     8.78          2478            5.07                     17.9
4       825     8.79          3303            4.39                     15.5
            sum  count      mean
user_type                       
A          1375   5421  0.253643
B          1695   5435  0.311868
A/A p_value: 0.0 | significant? True
SRM p_value: 0.4336 | problem? False


           sum  count      mean
user_type                      
A          392   1629  0.240638
B          471   1674  0.281362
A/B p_value: 0.0077 | significant? True
Lift: 4.07 pp | 95% CI: [ 1.08 , 7.06 ] pp


Decision about release of the new ML system

In [351]:
#A/A test: failed.The p-value is practically 0 far below 0.05, so we reject H0.
#SRM test. The group sizes are 1,629 and 1,674, and the p-value is 0.43, above 0.001. The sizes are fine, so the problem is not the group sizes.
#A/B test. The p-value is 0.0077, so on its face the result is significant. The lift is +4.07 pp
#However we can't trust the A/B result. B was ahead by about 5.8 pp before the experiment
#Decision: do not release based on this test. The experiment is invalid because it failed the A/A test.